# 🦙 Ollama Playground — run any model + test it in the browser

Pulls an **Ollama** model on Colab and gives you a **web chat UI (Gradio)** with a public link, so
you can test the model right in your browser (phone too).

Default model is the **1.5B abliterated DeepSeek-R1** (`huihui_ai/deepseek-r1-abliterated:1.5b`) —
small & fast, runs even on CPU. Change `MODEL` to any Ollama tag.

> ⚖️ Abliterated models have the refusal direction removed — for safety-research testing only.

## 1 · (Optional) GPU — small models run on CPU; a T4 just makes it faster

In [ ]:
!nvidia-smi || echo "no GPU detected — small models still run fine on CPU"

## 2 · Config

In [ ]:
#@title 🔧 Config { display-mode: "form" }
MODEL = "huihui_ai/deepseek-r1-abliterated:1.5b"  #@param {type:"string"}
SHARE = True  #@param {type:"boolean"}
#@markdown - **MODEL** — any Ollama tag, e.g. `llama3.2:3b`, `qwen2.5:1.5b`, `deepseek-r1:8b`, `huihui_ai/deepseek-r1-abliterated:8b`.
#@markdown - **SHARE** — also expose a public `*.gradio.live` link to open on any device.
print("MODEL =", MODEL, "| SHARE =", SHARE)

## 3 · Install (Ollama + Gradio)
The Ollama installer needs `zstd` on Colab — install it first.

In [ ]:
import subprocess, sys
def sh(cmd):
    print("+", cmd); subprocess.run(cmd, shell=True, check=False)
sh("apt-get -qq install -y zstd")                     # ollama installer needs zstd to extract
sh("curl -fsSL https://ollama.com/install.sh | sh")
sh(f"{sys.executable} -m pip install -q ollama gradio")
print("\u2705 install done")

## 4 · Start Ollama + pull the model

In [ ]:
import os, time, subprocess, requests
env = os.environ.copy(); env["OLLAMA_HOST"] = "127.0.0.1:11434"

def up():
    try: return requests.get("http://127.0.0.1:11434/api/tags", timeout=3).status_code == 200
    except Exception: return False

if not up():
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(60):
        if up(): break
        time.sleep(2)
print("ollama up:", up())

print("pulling", MODEL, "... (first time downloads the weights)")
subprocess.run(["ollama", "pull", MODEL], check=True, env=env)
print("\u2705 pulled", MODEL)

## 5 · Quick test (no UI)

In [ ]:
import ollama
r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Say hello in one short sentence."}])
print(r["message"]["content"])

## 6 · Web chat UI (Gradio) — test in the browser
Launches an interactive chat. With `SHARE=True` you also get a public `*.gradio.live` link.
DeepSeek-R1 models stream their `<think>…</think>` reasoning before the final answer.

In [ ]:
import gradio as gr, ollama

def chat(message, history):
    msgs = [{"role": m["role"], "content": m["content"]} for m in history]
    msgs.append({"role": "user", "content": message})
    out = ""
    for chunk in ollama.chat(model=MODEL, messages=msgs, stream=True):
        out += chunk["message"]["content"]
        yield out

demo = gr.ChatInterface(
    chat,
    type="messages",
    title=f"\U0001f999 {MODEL}",
    description="Ollama playground · streaming · abliteration test",
)
demo.launch(share=SHARE)

## Notes
- The `*.gradio.live` share link is **public for ~72h** — anyone with it can chat with the model.
  Stop the Gradio cell (or disconnect the runtime) when done.
- Switch models: change `MODEL` in §2 and re-run §4 + §6. Ollama keeps pulled models on disk for the session.
- Want an OpenAI-compatible HTTP endpoint instead of a web UI? Ollama already serves one at
  `http://127.0.0.1:11434/v1` — see `Z3r0_serve.ipynb` for an authenticated public version.